In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PETASE_INPUT_PATH = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/BetaTest/Ensemble/petase_full_rank_vespa_updated.csv"
)

PETASE_OUTPUT_PATH = PETASE_INPUT_PATH.with_name(
    "petase_full_rank_ensemble_updated.csv"
)

df = pd.read_csv(PETASE_INPUT_PATH)

print("Loaded:", PETASE_INPUT_PATH)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
rank_cols_to_drop = [
    c for c in df.columns
    if c.endswith("_rank")
    or c.endswith("_rank_norm")
    or c.endswith("_oriented")
    or c in ["full_rank", "full_rank_position", "final_score"]
]

df = df.drop(columns=rank_cols_to_drop, errors="ignore").copy()

print("Dropped columns:")
print(rank_cols_to_drop)

print("\nRemaining columns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
score_column_map = {
    "ESM2": "esm_index",
    "esmIF_index": "esmIF_index",
    "prosst": "prosst",
    "esm3_index": "esm3_index",
    "VESPAl": "vespa_score",
    "eve_index": "eve_index",
}

missing_score_cols = [
    col for col in score_column_map.values()
    if col not in df.columns
]

if missing_score_cols:
    raise KeyError(f"Missing score columns: {missing_score_cols}")

print("Score columns found:")
for model, col in score_column_map.items():
    print(f"{model:12s} <- {col}")

In [ ]:
# Optimierte Gewichte aus DLG4.
# PETase hat kein Rosetta/ddg.
# Daher wird der ddg-Anteil auf ESM2 addiert.

WEIGHTS_RAW = {
    "ESM2": 0.2746857597730656,
    "esmIF_index": 0.25035081868305303,
    "prosst": 0.19495858811260022,
    "esm3_index": 0.17374025655782047,
    "VESPAl": 0.10554910473846794,
    "eve_index": 0.000696004140803299,
    "ddg": 0.00001946799418932305,
}

WEIGHTS = {
    "ESM2": WEIGHTS_RAW["ESM2"] + WEIGHTS_RAW["ddg"],
    "esmIF_index": WEIGHTS_RAW["esmIF_index"],
    "prosst": WEIGHTS_RAW["prosst"],
    "esm3_index": WEIGHTS_RAW["esm3_index"],
    "VESPAl": WEIGHTS_RAW["VESPAl"],
    "eve_index": WEIGHTS_RAW["eve_index"],
}

MODEL_SIGNS = {
    "ESM2": 1,
    "esmIF_index": 1,
    "prosst": 1,
    "esm3_index": 1,
    "VESPAl": -1,
    "eve_index": 1,
}

weight_sum = sum(WEIGHTS.values())

print("Weight sum before normalization:", weight_sum)

if not np.isclose(weight_sum, 1.0):
    WEIGHTS = {k: v / weight_sum for k, v in WEIGHTS.items()}
    print("Weights were normalized.")

print("Weight sum after normalization:", sum(WEIGHTS.values()))

display(
    pd.DataFrame(
        {
            "model": list(WEIGHTS.keys()),
            "weight": list(WEIGHTS.values()),
            "sign": [MODEL_SIGNS[m] for m in WEIGHTS.keys()],
        }
    )
)

In [ ]:
df_ranked = df.copy()

for model, source_col in score_column_map.items():
    sign = MODEL_SIGNS[model]
    oriented_col = f"{model}_oriented"

    df_ranked[oriented_col] = (
        pd.to_numeric(df_ranked[source_col], errors="coerce") * sign
    )

oriented_cols = [f"{model}_oriented" for model in score_column_map.keys()]

display(
    df_ranked[
        ["sequence", "variant"]
        + list(score_column_map.values())
        + oriented_cols
    ].head()
)

In [ ]:
for model in WEIGHTS.keys():
    oriented_col = f"{model}_oriented"
    rank_col = f"{model}_rank_norm"

    x = pd.to_numeric(df_ranked[oriented_col], errors="coerce")
    valid = x.notna() & np.isfinite(x)
    n = int(valid.sum())

    df_ranked[rank_col] = np.nan

    if n == 0:
        print(f"[WARN] No valid values for {model}")
        continue

    ranks = x[valid].rank(method="average", ascending=False)

    if n == 1:
        df_ranked.loc[valid, rank_col] = 0.0
    else:
        df_ranked.loc[valid, rank_col] = (ranks - 1) / (n - 1)

    print(f"{model:12s}: ranked {n} values")

In [ ]:
rank_norm_cols = {
    model: f"{model}_rank_norm"
    for model in WEIGHTS.keys()
}

print("Rank columns:")
print(rank_norm_cols)

missing_rank_cols = [
    col for col in rank_norm_cols.values()
    if col not in df_ranked.columns
]

if missing_rank_cols:
    raise KeyError(f"Missing rank columns: {missing_rank_cols}")

rank_check = df_ranked[list(rank_norm_cols.values())].isna().sum()

print("NaNs per rank column:")
print(rank_check)

In [ ]:
rank_matrix = df_ranked[list(rank_norm_cols.values())].apply(
    pd.to_numeric,
    errors="coerce",
)

complete = np.all(np.isfinite(rank_matrix.to_numpy(dtype=float)), axis=1)

df_ranked["full_rank"] = np.nan

df_ranked.loc[complete, "full_rank"] = sum(
    WEIGHTS[model] * df_ranked.loc[complete, rank_norm_cols[model]]
    for model in WEIGHTS.keys()
)

print("Rows with complete ensemble ranks:", int(complete.sum()))
print("Rows with missing full_rank:", df_ranked["full_rank"].isna().sum())

display(
    df_ranked.sort_values("full_rank", ascending=True)[
        ["sequence", "variant", "full_rank"]
        + list(score_column_map.values())
    ].head(20)
)

In [ ]:
df_ranked["full_rank_position"] = df_ranked["full_rank"].rank(
    method="average",
    ascending=True,
)

display(
    df_ranked.sort_values("full_rank", ascending=True)[
        ["sequence", "variant", "full_rank", "full_rank_position"]
    ].head(20)
)

In [ ]:
helper_cols = [
    c for c in df_ranked.columns
    if c.endswith("_oriented") or c.endswith("_rank_norm")
]

df_final = df_ranked.drop(columns=helper_cols, errors="ignore").copy()

print("Final columns:")
print(df_final.columns.tolist())

display(df_final.head())

In [ ]:
df_final.to_csv(PETASE_OUTPUT_PATH, index=False)

print("Saved PETase ensemble file to:")
print(PETASE_OUTPUT_PATH)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PETASE_ENSEMBLE_PATH = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/BetaTest/Ensemble/petase_full_rank_ensemble_updated.csv"
)

df = pd.read_csv(PETASE_ENSEMBLE_PATH)

print("Loaded:", PETASE_ENSEMBLE_PATH)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
df = df.drop(
    columns=[
        "final_score",
        "final_score_high_is_good",
    ],
    errors="ignore",
).copy()

print("Columns after dropping old score columns:")
print(df.columns.tolist())

In [ ]:
valid = df["full_rank"].notna() & np.isfinite(df["full_rank"])

min_rank = df.loc[valid, "full_rank"].min()
max_rank = df.loc[valid, "full_rank"].max()

df["final_score"] = np.nan

if np.isclose(min_rank, max_rank):
    df.loc[valid, "final_score"] = 1.0
else:
    df.loc[valid, "final_score"] = 1.0 - (
        (df.loc[valid, "full_rank"] - min_rank) / (max_rank - min_rank)
    )

print("full_rank range:")
print("min_rank:", min_rank)
print("max_rank:", max_rank)

print("\nfinal_score summary:")
print(df["final_score"].describe())

display(
    df.sort_values("full_rank", ascending=True)[
        ["sequence", "variant", "full_rank", "final_score"]
    ].head(20)
)

In [ ]:
PETASE_ENSEMBLE_CORRECTED_PATH = PETASE_ENSEMBLE_PATH.with_name(
    "petase_full_rank_ensemble_updated_correct_score.csv"
)

df.to_csv(PETASE_ENSEMBLE_CORRECTED_PATH, index=False)

print("Saved corrected ensemble file:")
print(PETASE_ENSEMBLE_CORRECTED_PATH)

In [ ]:
PETASE_TEST_INPUT_PATH = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/BetaTest/Ensemble/predictive-pet-zero-shot-test-2025_filled.csv"
)

PETASE_TEST_OUTPUT_PATH = PETASE_TEST_INPUT_PATH.with_name(
    "predictive-pet-zero-shot-test-2025_filled_updated_correct_score.csv"
)

df_pred = pd.read_csv(PETASE_TEST_INPUT_PATH)

print("Loaded prediction template:")
print(PETASE_TEST_INPUT_PATH)
print("Shape:", df_pred.shape)
print("Columns:")
print(df_pred.columns.tolist())

display(df_pred.head())

In [ ]:
high_WTs = ["ARSGPFSVSEENVSRLSASGFGGGTIYYPRENNTYGAVAISPGYTGTEASIAWLGERIASHGFVVITIDTITTLDQPDSRAEQLNAALNHMINRASSTVRSRIDSSRLAVMGHSMGGGGSLRLASQRPDLKAAIPLTPWHLNKNWSSVTVPTLIIGADLDTIAPVATHAKPFYNSLPSSISKAYLELDGATHFAPNIPNKIIGKYSVAWLKRFVDNDTRYTQFLCPGPRDGLFGEVEEYRSTCPF",
            "ASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVGAIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAALRQVASLNGTSSSPIYGKVDTARMGVMGWSMGGGGSLISAANNPSLKAAAPQAPWDSSTNFSSVTVPTLIFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMKRFMDNDTRYSTFACENPNSTAVSDFRTANCSLED",
            "ASRGPFSYQSFTVSRPSGYRAGTVYYPTNAGGPVGAIAIVPGFTARQSSINWWGPRLASHGFVVITIDTNSTLDQPDSRSRQQMAALSQVATLSRTSSSPIYNKVDTSRLGVMGWSMGGGGSLISARNNPSIKAAAPQAPWSASKNFSSLTVPTLIIACENDTIAPVNQHADTFYDSMSRNPREFLEINNGSHSCANSGNSNQALLGKKGVAWMKRFMDNDRRYTSFACSNPNSYNVSDFRVAACN"]

In [ ]:
high_WT_score = 0.99
other_WTs_score = 0.80

print("Number of high WTs:", len(high_WTs))

In [ ]:
score_by_sequence = (
    df.dropna(subset=["final_score"])
    .sort_values("final_score", ascending=False)
    .drop_duplicates(subset=["sequence"], keep="first")
    .set_index("sequence")["final_score"]
    .to_dict()
)

print("Number of scored sequences:", len(score_by_sequence))

In [ ]:
target_cols = [
    "activity_1 (μmol [TPA]/min·mg [E])",
    "activity_2 (μmol [TPA]/min·mg [E])",
    "expression (mg/mL)",
]

for col in target_cols:
    if col not in df_pred.columns:
        df_pred[col] = np.nan

print("Target columns:")
print(target_cols)

In [ ]:
for idx, row in df_pred.iterrows():
    seq = row["sequence"]

    if seq in high_WTs:
        fill_value = high_WT_score

    elif seq in score_by_sequence:
        fill_value = score_by_sequence[seq]

    else:
        fill_value = other_WTs_score

    for col in target_cols:
        df_pred.loc[idx, col] = fill_value

display(df_pred.head())
print(df_pred[target_cols].describe())

In [ ]:
print("Best variants according to corrected final_score:")
display(
    df.sort_values("final_score", ascending=False)[
        ["sequence", "variant", "full_rank", "final_score"]
    ].head(20)
)

print("Worst variants according to corrected final_score:")
display(
    df.sort_values("final_score", ascending=True)[
        ["sequence", "variant", "full_rank", "final_score"]
    ].head(20)
)

In [ ]:
df_pred.to_csv(PETASE_TEST_OUTPUT_PATH, index=False)

print("Saved corrected PETase submission:")
print(PETASE_TEST_OUTPUT_PATH)

# Comparison PETase submissions

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr

SUBMISSIONS_DIR = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/petase_zero_shot_submissions"
)

# Deine gerade gespeicherte korrigierte Submission
OUR_SUBMISSION_PATH = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/BetaTest/Ensemble/predictive-pet-zero-shot-test-2025_filled_updated_correct_score.csv"
)

OUT_DIR = SUBMISSIONS_DIR / "comparison_with_corrected_ours"
OUT_DIR.mkdir(parents=True, exist_ok=True)

target_cols = [
    "activity_1 (μmol [TPA]/min·mg [E])",
    "activity_2 (μmol [TPA]/min·mg [E])",
    "expression (mg/mL)",
]

our_df = pd.read_csv(OUR_SUBMISSION_PATH)
submission_files = sorted(SUBMISSIONS_DIR.glob("*.csv"))

print("Our corrected submission:", OUR_SUBMISSION_PATH)
print("Our shape:", our_df.shape)
print("Number of anonymous submission files:", len(submission_files))

display(our_df.head())

In [ ]:
def normalize_submission_columns(df):
    df = df.copy()
    rename_map = {}

    for col in df.columns:
        col_clean = (
            str(col)
            .strip()
            .lower()
            .replace("µ", "μ")
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
            .replace(".", "")
        )

        if col_clean == "sequence":
            rename_map[col] = "sequence"

        elif "activity1" in col_clean:
            rename_map[col] = "activity_1 (μmol [TPA]/min·mg [E])"

        elif "activity2" in col_clean:
            rename_map[col] = "activity_2 (μmol [TPA]/min·mg [E])"

        elif "expression" in col_clean:
            rename_map[col] = "expression (mg/mL)"

    return df.rename(columns=rename_map)

In [ ]:
def aggregate_submission_by_sequence(df, cols):
    """
    Reduziert auf eindeutige Sequenzen.
    Falls eine Sequenz mehrfach vorkommt, werden Scores gemittelt.
    """
    tmp = df[["sequence"] + cols].copy()

    for col in cols:
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    return tmp.groupby("sequence", as_index=False)[cols].mean()


def safe_spearman(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")

    mask = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 3:
        return np.nan

    if x[mask].nunique() <= 1 or y[mask].nunique() <= 1:
        return np.nan

    return spearmanr(x[mask], y[mask]).correlation


def safe_pearson(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")

    mask = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 3:
        return np.nan

    if x[mask].nunique() <= 1 or y[mask].nunique() <= 1:
        return np.nan

    return pearsonr(x[mask], y[mask])[0]


def top_k_overlap(df, col_a, col_b, k):
    a = df[["sequence", col_a]].dropna().copy()
    b = df[["sequence", col_b]].dropna().copy()

    if len(a) == 0 or len(b) == 0:
        return np.nan

    k = min(k, len(a), len(b))

    top_a = set(
        a.sort_values(col_a, ascending=False)
        .head(k)["sequence"]
    )

    top_b = set(
        b.sort_values(col_b, ascending=False)
        .head(k)["sequence"]
    )

    return len(top_a & top_b) / k

In [ ]:
our_df = normalize_submission_columns(our_df)

missing_ours = [c for c in ["sequence"] + target_cols if c not in our_df.columns]
if missing_ours:
    raise KeyError(f"Missing columns in our submission: {missing_ours}")

our_agg = aggregate_submission_by_sequence(our_df, target_cols)

print("Original our_df shape:", our_df.shape)
print("Aggregated our_df shape:", our_agg.shape)
print("Duplicated sequences in our_df:", our_df["sequence"].duplicated().sum())

display(our_agg.head())

In [ ]:
results = []

for path in submission_files:
    team_df_raw = pd.read_csv(path)
    team_df = normalize_submission_columns(team_df_raw)

    if "sequence" not in team_df.columns:
        print(f"Skipping {path.name}: no sequence column")
        continue

    available_target_cols = [
        col for col in target_cols
        if col in team_df.columns and col in our_agg.columns
    ]

    if len(available_target_cols) == 0:
        print(f"Skipping {path.name}: no matching target columns")
        print("Original columns:", team_df_raw.columns.tolist())
        print("Normalized columns:", team_df.columns.tolist())
        continue

    our_agg_tmp = aggregate_submission_by_sequence(our_df, available_target_cols)
    team_agg = aggregate_submission_by_sequence(team_df, available_target_cols)

    merged = our_agg_tmp.merge(
        team_agg,
        on="sequence",
        how="inner",
        suffixes=("_ours", "_team"),
        validate="one_to_one",
    )

    row = {
        "submission_file": path.name,
        "n_overlap": len(merged),
        "available_readouts": ", ".join(available_target_cols),
        "n_available_readouts": len(available_target_cols),
        "n_team_rows_original": len(team_df_raw),
        "n_team_sequences_unique": team_agg["sequence"].nunique(),
        "n_team_duplicate_sequences": int(team_df["sequence"].duplicated().sum()),
    }

    for col in target_cols:
        if col not in available_target_cols:
            row[f"{col}_spearman"] = np.nan
            row[f"{col}_pearson"] = np.nan
            row[f"{col}_top10pct_overlap"] = np.nan
            row[f"{col}_top5pct_overlap"] = np.nan
            row[f"{col}_top100_overlap"] = np.nan
            continue

        ours_col = f"{col}_ours"
        team_col = f"{col}_team"

        row[f"{col}_spearman"] = safe_spearman(
            merged[ours_col],
            merged[team_col],
        )

        row[f"{col}_pearson"] = safe_pearson(
            merged[ours_col],
            merged[team_col],
        )

        n = len(merged)

        row[f"{col}_top10pct_overlap"] = top_k_overlap(
            merged,
            ours_col,
            team_col,
            k=max(1, int(np.ceil(0.10 * n))),
        )

        row[f"{col}_top5pct_overlap"] = top_k_overlap(
            merged,
            ours_col,
            team_col,
            k=max(1, int(np.ceil(0.05 * n))),
        )

        row[f"{col}_top100_overlap"] = top_k_overlap(
            merged,
            ours_col,
            team_col,
            k=min(100, n),
        )

    results.append(row)

comparison_df = pd.DataFrame(results)

print("Comparison shape:", comparison_df.shape)
display(comparison_df.head())

print("Available readout counts:")
print(comparison_df["n_available_readouts"].value_counts())

In [ ]:
spearman_cols = [f"{col}_spearman" for col in target_cols]
pearson_cols = [f"{col}_pearson" for col in target_cols]
top10_cols = [f"{col}_top10pct_overlap" for col in target_cols]
top5_cols = [f"{col}_top5pct_overlap" for col in target_cols]
top100_cols = [f"{col}_top100_overlap" for col in target_cols]

comparison_df["mean_spearman"] = comparison_df[spearman_cols].mean(axis=1, skipna=True)
comparison_df["mean_pearson"] = comparison_df[pearson_cols].mean(axis=1, skipna=True)
comparison_df["mean_top10pct_overlap"] = comparison_df[top10_cols].mean(axis=1, skipna=True)
comparison_df["mean_top5pct_overlap"] = comparison_df[top5_cols].mean(axis=1, skipna=True)
comparison_df["mean_top100_overlap"] = comparison_df[top100_cols].mean(axis=1, skipna=True)

display(
    comparison_df.sort_values("mean_spearman", ascending=False)[
        [
            "submission_file",
            "n_overlap",
            "n_available_readouts",
            "available_readouts",
            "mean_spearman",
            "mean_pearson",
            "mean_top10pct_overlap",
            "mean_top5pct_overlap",
            "mean_top100_overlap",
        ]
    ].head(20)
)

In [ ]:
comparison_path = OUT_DIR / "submission_level_comparison_corrected.csv"
summary_path = OUT_DIR / "summary_statistics_corrected.csv"

comparison_df.to_csv(comparison_path, index=False)

metric_cols = [
    c for c in comparison_df.columns
    if c.endswith("_spearman")
    or c.endswith("_pearson")
    or c.endswith("_top10pct_overlap")
    or c.endswith("_top5pct_overlap")
    or c.endswith("_top100_overlap")
    or c.startswith("mean_")
]

summary = comparison_df[metric_cols].describe().T
summary.to_csv(summary_path)

print("Saved:")
print(comparison_path)
print(summary_path)

display(summary)

In [ ]:
print("Highest agreement by mean Spearman:")
display(
    comparison_df.sort_values("mean_spearman", ascending=False)[
        [
            "submission_file",
            "n_overlap",
            "n_available_readouts",
            "mean_spearman",
            "mean_pearson",
            "mean_top10pct_overlap",
            "mean_top5pct_overlap",
            "mean_top100_overlap",
        ]
    ].head(20)
)

print("Lowest agreement by mean Spearman:")
display(
    comparison_df.sort_values("mean_spearman", ascending=True)[
        [
            "submission_file",
            "n_overlap",
            "n_available_readouts",
            "mean_spearman",
            "mean_pearson",
            "mean_top10pct_overlap",
            "mean_top5pct_overlap",
            "mean_top100_overlap",
        ]
    ].head(20)
)

In [ ]:
print("Highest agreement by mean NDCG top 10%:")
display(
    comparison_df.sort_values("mean_top10pct_overlap", ascending=False)[
        [
            "submission_file",
            "n_overlap",
            "n_available_readouts",
            "mean_spearman",
            "mean_pearson",
            "mean_top10pct_overlap",
            "mean_top5pct_overlap",
            "mean_top100_overlap",
        ]
    ].head(20)
)

print("Lowest agreement by mean Spearman:")
display(
    comparison_df.sort_values("mean_top10pct_overlap", ascending=True)[
        [
            "submission_file",
            "n_overlap",
            "n_available_readouts",
            "mean_spearman",
            "mean_pearson",
            "mean_top10pct_overlap",
            "mean_top5pct_overlap",
            "mean_top100_overlap",
        ]
    ].head(20)
)